# TxSON Data Pipeline — Script Demo

This notebook demonstrates each script in the `scripts/` folder in the order they are used in the data pipeline:

1. **`soil_or_met.py`** — classify a raw `.dat` file as soil, met, or unknown
2. **`read_data.py`** — read a raw `.dat` file into a DatetimeIndexed DataFrame
3. **`dup_cleaner.py`** — resolve duplicate timestamps
4. **`time_cleaner.py`** — fill in missing hourly timestamps with NaN
5. **`get_data_dict.py`** — run the full pipeline over a folder of raw files

In [1]:
# Shared setup — edit this path to point to your raw data folder
DATA_FOLDER = r"C:\pyt\tx-soil-moisture\datasets\TxSON_data_2026-02-24"

import os

# lets first manually find met and soil files before we automate the process using a more principled and bulletproof approach in the next cell

# Pick one example soil file and one example met file for the single-file demos
all_files = [f for f in os.listdir(DATA_FOLDER) if f.endswith(".dat")]

# A met file typically has "_met" in the name
example_met_file  = next((os.path.join(DATA_FOLDER, f) for f in all_files if "_met" in f.lower()), None)
example_soil_file = next((os.path.join(DATA_FOLDER, f) for f in all_files if "_met" not in f.lower()), None)

print(f"Met  file: {os.path.basename(example_met_file)}")
print(f"Soil file: {os.path.basename(example_soil_file)}")

Met  file: CB01_met.dat
Soil file: CB01.dat


---
## 1. `soil_or_met.py`

`SoilOrMet.determine_data_file()` inspects the first 10 lines of a `.dat` file and classifies it as `'soil'`, `'met'`, or `'unknown'` based on:
- presence of a `YYYY-MM-DD HH:MM:SS` timestamp
- overlap with the expected soil / met column headers

In [2]:
from soil_or_met import SoilOrMet

SOIL_HEADER = "Date,Ppt,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Flag"
MET_HEADER  = "TIMESTAMP,RECORD,Rain_mm_Tot,AirTC_Avg,RH,WS_ms_S_WVT,WindDir_D1_WVT,SlrW_Avg,ETos,Rso"

classifier = SoilOrMet(
    current_met_header=MET_HEADER,
    current_soil_header=SOIL_HEADER,
    min_soil_features=9, # in almost all soil files, there are 11 features, but some are missing the 50 cm depth and thus have only 9 features.
    min_met_features=10,
)

print(f"Met  file classified as: {classifier.determine_data_file(example_met_file)}")
print(f"Soil file classified as: {classifier.determine_data_file(example_soil_file)}")

Met  file classified as: met
Soil file classified as: soil


---
## 2. `read_data.py`

`file_to_indexed_df(file_path, is_soil_or_met)` is the main entry point. Internally it calls:
- `read_soil()` — skips 5 header lines, uses `Date` as the index
- `read_met()` — skips 6 header lines, drops `RECORD`, standardises column names, uses `TIMESTAMP` (renamed `Date`) as the index
- `check_for_NaT()` — warns if any timestamps could not be parsed

Both readers sort the index stably and coerce all measurement columns to numeric.

In [9]:
from read_data import file_to_indexed_df

soil_raw = file_to_indexed_df(example_soil_file, is_soil_or_met="soil")
print(f"Soil — shape: {soil_raw.shape}, index type: {type(soil_raw.index).__name__}")
soil_raw.head()

Soil — shape: (85252, 10), index type: DatetimeIndex


,Ppt,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Flag
Date,,,,,,,,,,
2014-09-29 18:00:00,7.87,0.080,0.113,0.072,0.049,36.01,33.13,30.73,27.95,0
2014-09-29 19:00:00,0.00,0.079,0.113,0.073,0.049,34.23,33.07,30.92,27.93,0
2014-09-29 20:00:00,0.00,0.078,0.112,0.073,0.049,31.69,32.50,30.92,27.84,0
2014-09-29 21:00:00,0.00,0.077,0.111,0.073,0.049,29.41,31.69,30.79,27.79,0
2014-09-29 22:00:00,0.00,0.076,0.110,0.072,0.049,27.72,30.79,30.52,27.75,0


In [10]:
met_raw = file_to_indexed_df(example_met_file, is_soil_or_met="met")
print(f"Met  — shape: {met_raw.shape}, index type: {type(met_raw.index).__name__}")
met_raw.head()

Met  — shape: (15598, 8), index type: DatetimeIndex


,Ppt,Tair,RH,Wind speed,Wind direction,Srad,ETos,Rso
Date,,,,,,,,
2022-06-08 12:00:00,0.0,33.16,36.49,1.191,168.20,990.0,0.503,3.594
2022-06-08 13:00:00,0.0,34.30,29.99,0.745,189.30,1014.0,0.800,3.503
2022-06-08 14:00:00,0.0,35.98,28.25,0.677,126.60,981.0,0.787,3.204
2022-06-08 15:00:00,0.0,37.27,28.61,1.997,90.30,911.0,0.804,2.699
2022-06-08 16:00:00,0.0,36.35,30.99,3.137,77.67,757.6,0.727,2.034


---
## 3. `dup_cleaner.py`

`dup_cleaner(df)` resolves duplicate timestamps in three passes (all keep the first occurrence):

| Case | Condition |
|---|---|
| 1 | Same timestamp + same measurements + same flag |
| 2 | Same timestamp + same measurements + different flag |
| 3 | Same timestamp + different measurements |

Requires a `DatetimeIndex`. Returns a deduplicated DataFrame with the `DatetimeIndex` restored.

In [5]:
from dup_cleaner import dup_cleaner

n_before = soil_raw.index.duplicated().sum()
soil_deduped = dup_cleaner(soil_raw)
n_after = soil_deduped.index.duplicated().sum()

print(f"Soil — duplicate timestamps before: {n_before}, after: {n_after}")
print(f"Rows removed: {len(soil_raw) - len(soil_deduped)}")

Soil — duplicate timestamps before: 4077, after: 0
Rows removed: 4077


---
## 4. `time_cleaner.py`

`time_cleaner(df)` fills gaps in an hourly `DatetimeIndex` by reindexing over the full `date_range` from the first to last timestamp. Missing rows are inserted with `NaN` measurements.

Must be run **after** `dup_cleaner` (requires a unique `DatetimeIndex`).

In [6]:
from time_cleaner import time_cleaner

soil_clean = time_cleaner(soil_deduped)

expected_hours = int((soil_clean.index.max() - soil_clean.index.min()).total_seconds() / 3600) + 1
n_filled = soil_clean.isna().any(axis=1).sum() - soil_deduped.isna().any(axis=1).sum()

print(f"Soil — rows before: {len(soil_deduped)}, after: {len(soil_clean)}")
print(f"Expected hourly timestamps: {expected_hours}")
print(f"Timestamps filled in with NaN: {len(soil_clean) - len(soil_deduped)}")

Soil — rows before: 81175, after: 83007
Expected hourly timestamps: 83007
Timestamps filled in with NaN: 1832


---
## 5. `get_data_dict.py`

`data_ingest` runs the full pipeline over every `.dat` file in a folder:

1. `open_df()` — classifies each file with `SoilOrMet`, reads it with `read_data`
2. `prewash_data()` — applies `dup_cleaner` then `time_cleaner` to every station (when `prewash_the_data=True`)
3. `clean_data()` — (*not yet implemented*) will handle nonsensical values
4. `get_data_dict()` — orchestrates all of the above and returns `(met_dict, soil_dict)`

Keys in each dict are station names derived from the filename (e.g. `"CB01_met"`, `"CB01"`).

In [7]:
from get_data_dict import data_ingest

ingest = data_ingest(
    input_data_folder=DATA_FOLDER,
    prewash_the_data=True,
    clean_the_data=False,
)

met_dict, soil_dict = ingest.get_data_dict()

print(f"Met  stations loaded : {len(met_dict)}  — {list(met_dict.keys())}")
print(f"Soil stations loaded : {len(soil_dict)} — {list(soil_dict.keys())}")

Met  stations loaded : 6  — ['CB01_met', 'CB04_met', 'CB06_met', 'FD02_met', 'FD03_met', 'WC05_met']
Soil stations loaded : 33 — ['CB01', 'CB04', 'CB06', 'CB07', 'CB09', 'CB10', 'CB15', 'CB19', 'CB20', 'CB25', 'CB26', 'CB27', 'CB31', 'CB32', 'CB33', 'FD02', 'FD03', 'FD08', 'FD11', 'FD12', 'FD13', 'FD14', 'FD16', 'FD17', 'FD18', 'FD21', 'FD22', 'FD23', 'FD24', 'FD28', 'FD29', 'FD30', 'WC05']


In [8]:
# Inspect one station from each dict
first_met_key  = next(iter(met_dict))
first_soil_key = next(iter(soil_dict))

print(f"=== Met  [{first_met_key}] — {met_dict[first_met_key].shape} ===")
met_dict[first_met_key].head()

print(f"\n=== Soil [{first_soil_key}] — {soil_dict[first_soil_key].shape} ===")
soil_dict[first_soil_key].head()

=== Met  [CB01_met] — (15597, 8) ===

=== Soil [CB01] — (83007, 10) ===


,Ppt,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Flag
2014-09-29 18:00:00,7.87,0.080,0.113,0.072,0.049,36.01,33.13,30.73,27.95,0
2014-09-29 19:00:00,0.00,0.079,0.113,0.073,0.049,34.23,33.07,30.92,27.93,0
2014-09-29 20:00:00,0.00,0.078,0.112,0.073,0.049,31.69,32.50,30.92,27.84,0
2014-09-29 21:00:00,0.00,0.077,0.111,0.073,0.049,29.41,31.69,30.79,27.79,0
2014-09-29 22:00:00,0.00,0.076,0.110,0.072,0.049,27.72,30.79,30.52,27.75,0
